<a href="https://colab.research.google.com/github/taehi5/Esaa_YB/blob/10%EC%A3%BC%EC%B0%A8--%EA%B3%BC%EC%A0%9C/ESAA_YB_Week10_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

`파이썬 머신러닝 완벽가이드 ch4. 9~10 pg.267~295`


In [1]:
%pip install xgboost==1.7.6
%pip install lightgbm==4.1.0


# 분류 실습 - 캐글 산탄데르 고객 만족 예측

## 1. 데이터와 평가 기준

- 익명 처리된 수치형 피처로 고객 만족 여부를 예측함.
- TARGET: 0은 만족, 1은 불만족 고객임.
- 원본은 76,020행 × 371열이며 ID와 TARGET을 제외하면 설명 변수는 369개임.
- 불만족 고객이 약 4%인 불균형 데이터임.

✅ ROC-AUC

- 양성 예측 확률을 이용해 모델이 두 클래스를 얼마나 잘 구분하는지 평가함.
- 정확도만 사용하면 모두 만족으로 예측하는 모델도 높은 점수를 받을 수 있음.
- `predict_proba(X)[:, 1]`로 불만족 확률을 추출함.


## 2. 데이터 전처리

- `info()`: 데이터 타입과 결측 여부를 확인함.
- `value_counts()`: 레이블 분포를 확인함.
- `describe()`: 피처의 범위와 이상한 값을 확인함.
- var3의 -999999는 예외값으로 보고 최빈값인 2로 변경함.
- ID는 식별자이므로 삭제하고, TARGET을 예측 대상 y로 분리함.

❗ 수치형 데이터에도 결측을 나타내는 특수한 숫자가 있을 수 있음.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings

warnings.filterwarnings('ignore')

cust_df = pd.read_csv('/content/train.csv', encoding='latin-1')
print('dataset shape:', cust_df.shape)
cust_df.head(3)

dataset shape: (76020, 371)


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.17,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.03,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.77,0


In [3]:
cust_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76020 entries, 0 to 76019
Columns: 371 entries, ID to TARGET
dtypes: float64(111), int64(260)
memory usage: 215.2 MB


In [4]:
print(cust_df['TARGET'].value_counts())

unsatisfied_cnt = cust_df[cust_df['TARGET'] == 1].TARGET.count()
total_cnt = cust_df.TARGET.count()

print('unsatisfied 비율은 {0:.2f}'.format(unsatisfied_cnt / total_cnt))

TARGET
0    73012
1     3008
Name: count, dtype: int64
unsatisfied 비율은 0.04


In [5]:
cust_df.describe()

,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
count,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,...,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,7.602000e+04,76020.000000
mean,75964.050723,-1523.199277,33.212865,86.208265,72.363067,119.529632,3.559130,6.472698,0.412946,0.567352,...,7.935824,1.365146,12.215580,8.784074,31.505324,1.858575,76.026165,56.614351,1.172358e+05,0.039569
std,43781.947379,39033.462364,12.956486,1614.757313,339.315831,546.266294,93.155749,153.737066,30.604864,36.513513,...,455.887218,113.959637,783.207399,538.439211,2013.125393,147.786584,4040.337842,2852.579397,1.826646e+05,0.194945
min,1.000000,-999999.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.163750e+03,0.000000
25%,38104.750000,2.000000,23.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.787061e+04,0.000000
50%,76043.000000,2.000000,28.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.064092e+05,0.000000
75%,113748.750000,2.000000,40.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.187563e+05,0.000000
max,151838.000000,238.000000,105.000000,210000.000000,12888.030000,21024.810000,8237.820000,11073.570000,6600.000000,6600.000000,...,50003.880000,20385.720000,138831.630000,91778.730000,438329.220000,24650.010000,681462.900000,397884.300000,2.203474e+07,1.000000


In [6]:
cust_df['var3'].replace(-999999, 2, inplace=True)
cust_df.drop('ID', axis=1, inplace=True)

# 피처 세트와 레이블 세트 분리. 레이블 칼럼은 DataFrame의 맨 마지막에 위치해 칼럼 위치 -1로 분리
X_features = cust_df.iloc[:, :-1]
y_labels = cust_df.iloc[:, -1]

print('피처 데이터 shape:{0}'.format(X_features.shape))

피처 데이터 shape:(76020, 369)


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels, test_size=0.2, random_state=0
)

train_cnt = y_train.count()
test_cnt = y_test.count()

print('학습 세트 Shape:{0}, 테스트 세트 Shape:{1}'.format(X_train.shape, X_test.shape))

print('학습 세트 레이블 값 분포 비율')
print(y_train.value_counts() / train_cnt)

print('\n테스트 세트 레이블 값 분포 비율')
print(y_test.value_counts() / test_cnt)

학습 세트 Shape:(60816, 369), 테스트 세트 Shape:(15204, 369)
학습 세트 레이블 값 분포 비율
TARGET
0    0.960964
1    0.039036
Name: count, dtype: float64

테스트 세트 레이블 값 분포 비율
TARGET
0    0.9583
1    0.0417
Name: count, dtype: float64


In [8]:
# X_train, y_train을 다시 학습과 검증 데이터 세트로 분리.
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.3, random_state=0
)

✅ 데이터 분할의 역할

- X_tr / y_tr: 모델 학습에 사용함.
- X_val / y_val: 조기 중단을 결정하는 검증 데이터임.
- X_test / y_test: 최종 성능 확인에 사용함.
- 교재의 분할 조건을 유지함. 불균형 데이터에서는 `stratify=y`로 비율을 유지하는 방식도 활용 가능함.


## 3. XGBoost 모델 학습

- n_estimators=500, learning_rate=0.05로 설정함.
- 검증 AUC가 100회 동안 개선되지 않으면 학습을 중단함.
- 최종 테스트 데이터는 조기 중단용 eval_set에 넣지 않음.

In [9]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# n_estimators는 500으로, random_state는 예제 수행 시마다 동일 예측 결과를 위해 설정.
xgb_clf = XGBClassifier(n_estimators=500, learning_rate=0.05, random_state=156)

# 성능 평가 지표를 auc로, 조기 중단 파라미터는 100으로 설정하고 학습 수행.
xgb_clf.fit(
    X_tr, y_tr,
    early_stopping_rounds=100,
    eval_metric="auc",
    eval_set=[(X_tr, y_tr), (X_val, y_val)]
)

xgb_roc_score = roc_auc_score(y_test, xgb_clf.predict_proba(X_test)[:, 1])
print('ROC AUC: {0:.4f}'.format(xgb_roc_score))

[0]	validation_0-auc:0.82179	validation_1-auc:0.80068
[1]	validation_0-auc:0.82347	validation_1-auc:0.80523
[2]	validation_0-auc:0.83178	validation_1-auc:0.81097
[3]	validation_0-auc:0.83401	validation_1-auc:0.81091
[4]	validation_0-auc:0.83443	validation_1-auc:0.81040
[5]	validation_0-auc:0.83570	validation_1-auc:0.81089
[6]	validation_0-auc:0.83597	validation_1-auc:0.81057
[7]	validation_0-auc:0.83643	validation_1-auc:0.81082
[8]	validation_0-auc:0.83682	validation_1-auc:0.81147
[9]	validation_0-auc:0.83769	validation_1-auc:0.81188
[10]	validation_0-auc:0.83770	validation_1-auc:0.81163
[11]	validation_0-auc:0.83911	validation_1-auc:0.81355
[12]	validation_0-auc:0.83976	validation_1-auc:0.81336
[13]	validation_0-auc:0.84038	validation_1-auc:0.81365
[14]	validation_0-auc:0.84176	validation_1-auc:0.81419
[15]	validation_0-auc:0.84306	validation_1-auc:0.81586
[16]	validation_0-auc:0.84343	validation_1-auc:0.81610
[17]	validation_0-auc:0.84373	validation_1-auc:0.81584
[18]	validation_0-au

** HyperOpt를 이용한 XGBoost 튜닝**

✅ 수행 과정

1. 하이퍼파라미터 검색 공간을 설정함.
2. 3폴드 교차 검증의 평균 AUC에 -1을 곱해 목적 함수에서 반환함.
3. fmin()으로 목적 함수를 50회 평가함.
4. 찾은 최적값을 적용하고 트리 개수를 500개로 늘려 다시 학습함.

- max_depth: 트리 깊이
- min_child_weight: 자식 노드 분할을 제한하는 가중치 합의 최소 기준
- colsample_bytree: 트리별로 사용하는 피처 비율
- learning_rate: 부스팅 학습률

❗ hp.quniform()의 반환값은 실수이므로 정수 파라미터는 int()로 바꿈. hp.uniform()은 **균등 분포**로 샘플링함.

In [10]:
from hyperopt import hp

# max_depth는 5에서 15까지 1 간격으로, min_child_weight는 1에서 6까지 1 간격으로
# colsample_bytree는 0.5에서 0.95 사이, learning_rate는 0.01에서 0.2 사이 균등 분포된 값으로 검색.
xgb_search_space = {
    'max_depth': hp.quniform('max_depth', 5, 15, 1),
    'min_child_weight': hp.quniform('min_child_weight', 1, 6, 1),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 0.95),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2)
}

In [11]:
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

# fmin()에서 호출 시 search_space 값으로 XGBClassifier 교차 검증 학습 후 -1 * roc_auc 평균 값을 반환.
def objective_func(search_space):
    xgb_clf = XGBClassifier(
        n_estimators=100,
        max_depth=int(search_space['max_depth']),
        min_child_weight=int(search_space['min_child_weight']),
        colsample_bytree=search_space['colsample_bytree'],
        learning_rate=search_space['learning_rate']
    )

    # 3개 k-fold 방식으로 평가된 roc_auc 지표를 담는 list
    roc_auc_list = []

    # 3개 k-fold 방식 적용
    kf = KFold(n_splits=3)

    # X_train을 다시 학습과 검증용 데이터로 분리
    for tr_index, val_index in kf.split(X_train):
        # kf.split(X_train)으로 추출된 학습과 검증 index 값으로 학습과 검증 데이터 세트 분리
        X_tr, y_tr = X_train.iloc[tr_index], y_train.iloc[tr_index]
        X_val, y_val = X_train.iloc[val_index], y_train.iloc[val_index]

        # early stopping은 30회로 설정하고 추출된 학습과 검증 데이터로 XGBClassifier 학습 수행.
        xgb_clf.fit(
            X_tr, y_tr,
            early_stopping_rounds=30,
            eval_metric="auc",
            eval_set=[(X_tr, y_tr), (X_val, y_val)]
        )

        # 1로 예측한 확률값 추출 후 roc_auc 계산하고 평균 roc_auc 계산을 위해 list에 결괏값 담음.
        score = roc_auc_score(y_val, xgb_clf.predict_proba(X_val)[:, 1])
        roc_auc_list.append(score)

    # 3개 k-fold로 계산된 roc_auc 값의 평균값을 반환하되,
    # HyperOpt는 목적함수의 최솟값을 위한 입력값을 찾으므로 -1을 곱한 뒤 반환.
    return -1 * np.mean(roc_auc_list)

In [ ]:
from hyperopt import fmin, tpe, Trials

trials = Trials()

# fmin() 함수를 호출. max_evals 지정된 횟수만큼 반복 후 목적함수의 최솟값을 가지는 최적 입력값 추출.
best = fmin(
    fn=objective_func,
    space=xgb_search_space,
    algo=tpe.suggest,
    max_evals=50,  # 최대 반복 횟수를 지정합니다.
    trials=trials,
    rstate=np.random.default_rng(seed=30)
)

print('best:', best)

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
[0]	validation_0-auc:0.83186	validation_1-auc:0.79893
[1]	validation_0-auc:0.82728	validation_1-auc:0.78934
[2]	validation_0-auc:0.84375	validation_1-auc:0.80250
[3]	validation_0-auc:0.84963	validation_1-auc:0.80549
[4]	validation_0-auc:0.85863	validation_1-auc:0.81080
[5]	validation_0-auc:0.86245	validation_1-auc:0.80900
[6]	validation_0-auc:0.86646	validation_1-auc:0.81127
[7]	validation_0-auc:0.87137	validation_1-auc:0.81318
[8]	validation_0-auc:0.87472	validation_1-auc:0.81570
[9]	validation_0-auc:0.87932	validation_1-auc:0.81832
[10]	validation_0-auc:0.87968	validation_1-auc:0.81602
[11]	validation_0-auc:0.88772	validation_1-auc:0.81754
[12]	validation_0-auc:0.89413	validation_1-auc:0.81904
[13]	validation_0-auc:0.89976	validation_1-auc:0.81644
[14]	validation_0-auc:0.90208	validation_1-auc:0.81969
[15]	validation_0-auc:0.90320	validation_1-auc:0.81714
[16]	validation_0-auc:0.90406	validation_1-auc:0.81605
[17]	validation_0-auc:0.90492	validatio

In [ ]:
# n_estimators를 500 증가 후 최적으로 찾은 하이퍼 파라미터를 기반으로 학습과 예측 수행.
xgb_clf = XGBClassifier(
    n_estimators=500,
    learning_rate=round(best['learning_rate'], 5),
    max_depth=int(best['max_depth']),
    min_child_weight=int(best['min_child_weight']),
    colsample_bytree=round(best['colsample_bytree'], 5)
)

# evaluation metric을 auc로, early stopping은 100으로 설정하고 학습 수행.
xgb_clf.fit(
    X_tr, y_tr,
    early_stopping_rounds=100,
    eval_metric="auc",
    eval_set=[(X_tr, y_tr), (X_val, y_val)]
)

xgb_roc_score = roc_auc_score(y_test, xgb_clf.predict_proba(X_test)[:, 1])
print('ROC AUC: {0:.4f}'.format(xgb_roc_score))

✅ 피처 중요도 (p.275)

- plot_importance()로 모델의 예측에 사용된 피처의 중요도를 확인함.
- 교재에서는 var38, var15가 상위 피처로 나타남.
- 중요도가 높다는 것이 인과관계나 피처의 실제 의미를 알려주는 것은 아님.

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
plot_importance(xgb_clf, ax=ax, max_num_features=20, height=0.4)

## 5. LightGBM 모델 학습과 튜닝

- XGBoost와 동일한 데이터에서 성능을 비교함.
- 리프 중심 분할을 사용하므로 num_leaves와 min_child_samples로 모델 복잡도를 조절함.
- LightGBM 4.1.0에서는 fit()의 `early_stopping_rounds` 대신 `early_stopping()` 콜백을 사용함.
- 교재 조건을 따라 subsample을 설정함. 기본 subsample_freq=0에서는 행 샘플링이 활성화되지 않으므로 실제 샘플링을 적용하려면 subsample_freq를 양수로 추가해야 함.


In [ ]:
from lightgbm import LGBMClassifier

lgbm_clf = LGBMClassifier(n_estimators=500)

eval_set = [(X_tr, y_tr), (X_val, y_val)]
lgbm_clf.fit(
    X_tr, y_tr,
    early_stopping_rounds=100,
    eval_metric="auc",
    eval_set=eval_set
)

lgbm_roc_score = roc_auc_score(y_test, lgbm_clf.predict_proba(X_test)[:, 1])
print('ROC AUC: {0:.4f}'.format(lgbm_roc_score))

In [ ]:
lgbm_search_space = {
    'num_leaves': hp.quniform('num_leaves', 32, 64, 1),
    'max_depth': hp.quniform('max_depth', 100, 160, 1),
    'min_child_samples': hp.quniform('min_child_samples', 60, 100, 1),
    'subsample': hp.uniform('subsample', 0.7, 1),
    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2)
}

In [ ]:
def objective_func(search_space):
    lgbm_clf = LGBMClassifier(
        n_estimators=100,
        num_leaves=int(search_space['num_leaves']),
        max_depth=int(search_space['max_depth']),
        min_child_samples=int(search_space['min_child_samples']),
        subsample=search_space['subsample'],
        learning_rate=search_space['learning_rate']
    )

    # 3개 k-fold 방식으로 평가된 roc_auc 지표를 담는 list
    roc_auc_list = []

    # 3개 k-fold 방식 적용
    kf = KFold(n_splits=3)

    # X_train을 다시 학습과 검증용 데이터로 분리
    for tr_index, val_index in kf.split(X_train):
        # kf.split(X_train)으로 추출된 학습과 검증 index 값으로 학습과 검증 데이터 세트 분리
        X_tr, y_tr = X_train.iloc[tr_index], y_train.iloc[tr_index]
        X_val, y_val = X_train.iloc[val_index], y_train.iloc[val_index]

        # early stopping은 30회로 설정하고 추출된 학습과 검증 데이터로 LGBMClassifier 학습 수행.
        lgbm_clf.fit(
            X_tr, y_tr,
            early_stopping_rounds=30,
            eval_metric="auc",
            eval_set=[(X_tr, y_tr), (X_val, y_val)]
        )

        # 1로 예측한 확률값 추출 후 roc_auc 계산하고 평균 roc_auc 계산을 위해 list에 결괏값 담음.
        score = roc_auc_score(y_val, lgbm_clf.predict_proba(X_val)[:, 1])
        roc_auc_list.append(score)

    # 3개 k-fold로 계산된 roc_auc 값의 평균값을 반환하되,
    # HyperOpt는 목적함수의 최솟값을 위한 입력값을 찾으므로 -1을 곱한 뒤 반환.
    return -1 * np.mean(roc_auc_list)

In [ ]:
from hyperopt import fmin, tpe, Trials

trials = Trials()

# fmin() 함수를 호출. max_evals 지정된 횟수만큼 반복 후 목적함수의 최솟값을 가지는 최적 입력값 추출.
best = fmin(
    fn=objective_func,
    space=lgbm_search_space,
    algo=tpe.suggest,
    max_evals=50,  # 최대 반복 횟수를 지정합니다.
    trials=trials,
    rstate=np.random.default_rng(seed=30)
)

print('best:', best)

In [ ]:
lgbm_clf = LGBMClassifier(
    n_estimators=500,
    num_leaves=int(best['num_leaves']),
    max_depth=int(best['max_depth']),
    min_child_samples=int(best['min_child_samples']),
    subsample=round(best['subsample'], 5),
    learning_rate=round(best['learning_rate'], 5)
)

# evaluation metric을 auc로, early stopping은 100으로 설정하고 학습 수행.
lgbm_clf.fit(
    X_tr, y_tr,
    early_stopping_rounds=100,
    eval_metric="auc",
    eval_set=[(X_tr, y_tr), (X_val, y_val)]
)

lgbm_roc_score = roc_auc_score(y_test, lgbm_clf.predict_proba(X_test)[:, 1])
print('ROC AUC: {0:.4f}'.format(lgbm_roc_score))

✅ 산탄데르 실습 정리

- 불균형 데이터는 정확도 외에 ROC-AUC로 구분 능력을 확인함.
- HyperOpt는 검증 성능을 기준으로 파라미터를 탐색하며, 탐색 횟수와 학습 시간이 증가함.
- 튜닝이 항상 큰 폭의 성능 향상을 보장하지는 않음.
- 교재 참고 AUC: XGBoost 0.8429 → 0.8457, LightGBM 0.8384 → 0.8446.
- 라이브러리·난수·학습 조건에 따라 직접 실행한 수치는 달라질 수 있음.

# 분류 실습 - 캐글 신용카드 사기 검출

## 1. 불균형 데이터와 샘플링

- Class: 0은 정상 거래, 1은 사기 거래임.
- 284,807건 중 사기 거래는 492건으로 약 0.172%임.
- 모든 거래를 정상으로 예측해도 정확도가 약 99.83%이므로 정확도만으로 평가하면 안 됨.

✅ 언더 샘플링

- 다수 클래스 데이터를 줄여 클래스 차이를 완화함.
- 학습량이 감소하지만 정상 거래의 중요한 정보가 사라질 수 있음.

✅ 오버 샘플링과 SMOTE

- 소수 클래스 데이터를 늘려 충분한 학습 데이터를 확보함.
- 단순 복제는 과적합으로 이어질 수 있음.
- SMOTE는 소수 클래스 표본과 가까운 이웃 표본 사이를 보간하여 새로운 표본을 생성함.
- `x_new = x + λ × (x_neighbor - x)`, 0 ≤ λ ≤ 1.

❗ 오버 샘플링은 반드시 학습 데이터에만 적용함. 검증·테스트 데이터는 원래 분포로 남김.

In [ ]:
!pip install imbalanced-learn

## 2. 데이터 로딩과 일차 가공

- V1~V28은 익명화된 피처임.
- Amount는 거래 금액, Time은 첫 거래로부터 경과한 시간임.
- 교재 흐름에 따라 Time을 제거하고 Class를 레이블로 분리함.
- stratify=y_target으로 학습/테스트의 사기 비율을 유지함.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")
%matplotlib inline

card_df = pd.read_csv('/content/creditcard.csv')
card_df.head(3)

In [ ]:
from sklearn.model_selection import train_test_split

# 인자로 입력받은 DataFrame을 복사한 뒤 Time 칼럼만 삭제하고 복사된 DataFrame 반환
def get_preprocessed_df(df=None):
    df_copy = df.copy()
    df_copy.drop('Time', axis=1, inplace=True)
    return df_copy

In [ ]:
# 사전 데이터 가공 후 학습과 테스트 데이터 세트를 반환하는 함수.
def get_train_test_dataset(df=None):
    # 인자로 입력된 DataFrame의 사전 데이터 가공이 완료된 복사 DataFrame 반환
    df_copy = get_preprocessed_df(df)

    # DataFrame의 맨 마지막 칼럼이 레이블, 나머지는 피처들
    X_features = df_copy.iloc[:, :-1]
    y_target = df_copy.iloc[:, -1]

    # train_test_split()으로 학습과 테스트 데이터 분할. stratify=y_target으로 Stratified 기반 분할
    X_train, X_test, y_train, y_test = train_test_split(
        X_features, y_target,
        test_size=0.3,
        random_state=0,
        stratify=y_target
    )

    # 학습과 테스트 데이터 세트 반환
    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = get_train_test_dataset(card_df)

In [ ]:
print('학습 데이터 레이블 값 비율')
print(y_train.value_counts() / y_train.shape[0] * 100)

print('테스트 데이터 레이블 값 비율')
print(y_test.value_counts() / y_test.shape[0] * 100)

## 3. 평가 함수와 기본 모델

✅ 평가 지표

- 오차 행렬: `[[TN, FP], [FN, TP]]`
- 정밀도 = TP / (TP + FP): 사기로 예측한 거래 중 실제 사기의 비율
- 재현율 = TP / (TP + FN): 실제 사기 중 찾아낸 비율
- F1: 정밀도와 재현율의 조화 평균
- ROC-AUC: 전체 임곗값에 걸친 구분 능력

❗ 사기 검출에서는 놓친 사기(FN)와 정상 거래를 사기로 잘못 예측한 경우(FP)를 함께 고려함.



In [ ]:
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix(y_test, pred)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)

    # ROC-AUC 추가
    roc_auc = roc_auc_score(y_test, pred_proba)

    print('오차 행렬')
    print(confusion)

    # ROC-AUC print 추가
    print(
        '정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f}, F1: {3:.4f}, AUC: {4:.4f}'.format(
            accuracy, precision, recall, f1, roc_auc
        )
    )

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_clf = LogisticRegression(max_iter=1000)
lr_clf.fit(X_train, y_train)
lr_pred = lr_clf.predict(X_test)
lr_pred_proba = lr_clf.predict_proba(X_test)[:, 1]

# 3장에서 사용한 get_clf_eval() 함수를 이용해 평가 수행.
get_clf_eval(y_test, lr_pred, lr_pred_proba)

In [ ]:
# 인자로 사이킷런의 Estimator 객체와 학습/테스트 데이터 세트를 입력받아서 학습/예측/평가 수행.
def get_model_train_eval(model, ftr_train=None, ftr_test=None, tgt_train=None, tgt_test=None):
    model.fit(ftr_train, tgt_train)
    pred = model.predict(ftr_test)
    pred_proba = model.predict_proba(ftr_test)[:, 1]
    get_clf_eval(tgt_test, pred, pred_proba)

In [ ]:
from lightgbm import LGBMClassifier

lgbm_clf = LGBMClassifier(
    n_estimators=1000,
    num_leaves=64,
    n_jobs=-1,
    boost_from_average=False
)

get_model_train_eval(
    lgbm_clf,
    ftr_train=X_train,
    ftr_test=X_test,
    tgt_train=y_train,
    tgt_test=y_test
)

In [ ]:
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix(y_test, pred)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred, zero_division=0)
    recall = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f}, F1: {3:.4f}, AUC: {4:.4f}'.format(
        accuracy, precision, recall, f1, roc_auc
    ))
    return dict(accuracy=accuracy, precision=precision, recall=recall, f1=f1, auc=roc_auc)

from sklearn.linear_model import LogisticRegression
lr_clf = LogisticRegression(max_iter=1000)
lr_clf.fit(X_train, y_train)
lr_pred = lr_clf.predict(X_test)
lr_pred_proba = lr_clf.predict_proba(X_test)[:, 1]
results = []
results.append(dict(stage='원본', model='LogisticRegression',
                    **get_clf_eval(y_test, lr_pred, lr_pred_proba)))

In [ ]:
def get_model_train_eval(model, ftr_train=None, ftr_test=None, tgt_train=None, tgt_test=None):
    model.fit(ftr_train, tgt_train)
    pred = model.predict(ftr_test)
    pred_proba = model.predict_proba(ftr_test)[:, 1]
    return get_clf_eval(tgt_test, pred, pred_proba)

lgbm_clf = LGBMClassifier(
    n_estimators=1000, num_leaves=64, n_jobs=-1, boost_from_average=False, verbosity=-1
)
results.append(dict(stage='원본', model='LightGBM', **get_model_train_eval(
    lgbm_clf, X_train, X_test, y_train, y_test
)))

- 반복되는 학습·예측·평가를 함수로 묶어 전처리 효과를 비교함.
- boost_from_average=False는 교재 실험 설정임. 불균형 데이터에서 항상 이 설정이 최적이라는 의미는 아님.

## 4. 데이터 분포 변환

1) Amount 분포 확인

- 대부분 작은 거래 금액이고 일부 큰 거래 금액이 존재해 오른쪽 꼬리가 긴 분포임.

2) StandardScaler

- 평균을 0, 표준편차를 1로 맞추는 표준화임.
- **표준화한다고 원래 분포가 정규 분포로 바뀌는 것은 아님.**

3) 로그 변환

- np.log1p(x)는 log(1+x)로, 0에서도 계산 가능함.
- 큰 값을 상대적으로 압축하여 치우친 분포를 완화함.
- 변환에 따른 성능 변화는 모델과 데이터에 따라 다름.

In [ ]:
import seaborn as sns

plt.figure(figsize=(8, 4))
plt.xticks(range(0, 30000, 1000), rotation=60)
sns.histplot(card_df['Amount'], bins=100, kde=True)
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

# 사이킷런의 StandardScaler를 이용해 정규 분포 형태로 Amount 피처값 변환하는 로직으로 수정.
def get_preprocessed_df(df=None):
    df_copy = df.copy()
    scaler = StandardScaler()
    amount_n = scaler.fit_transform(df_copy['Amount'].values.reshape(-1, 1))

    # 변환된 Amount를 Amount_Scaled로 피처명 변경 후 DataFrame 맨 앞 칼럼으로 입력
    df_copy.insert(0, 'Amount_Scaled', amount_n)

    # 기존 Time, Amount 피처 삭제
    df_copy.drop(['Time', 'Amount'], axis=1, inplace=True)

    return df_copy

In [ ]:
# Amount를 정규 분포 형태로 변환 후 로지스틱 회귀 및 LightGBM 수행.
X_train, X_test, y_train, y_test = get_train_test_dataset(card_df)

print('### 로지스틱 회귀 예측 성능 ###')
lr_clf = LogisticRegression(max_iter=1000)
get_model_train_eval(
    lr_clf,
    ftr_train=X_train,
    ftr_test=X_test,
    tgt_train=y_train,
    tgt_test=y_test
)

print('### LightGBM 예측 성능 ###')
lgbm_clf = LGBMClassifier(
    n_estimators=1000,
    num_leaves=64,
    n_jobs=-1,
    boost_from_average=False
)
get_model_train_eval(
    lgbm_clf,
    ftr_train=X_train,
    ftr_test=X_test,
    tgt_train=y_train,
    tgt_test=y_test
)

In [ ]:
def get_preprocessed_df(df=None):
    df_copy = df.copy()

    # 넘파이의 log1p()를 이용해 Amount를 로그 변환
    amount_n = np.log1p(df_copy['Amount'])
    df_copy.insert(0, 'Amount_Scaled', amount_n)
    df_copy.drop(['Time', 'Amount'], axis=1, inplace=True)

    return df_copy

In [ ]:
X_train, X_test, y_train, y_test = get_train_test_dataset(card_df)

print('### 로지스틱 회귀 예측 성능 ###')
get_model_train_eval(
    lr_clf,
    ftr_train=X_train,
    ftr_test=X_test,
    tgt_train=y_train,
    tgt_test=y_test
)

print('### LightGBM 예측 성능 ###')
get_model_train_eval(
    lgbm_clf,
    ftr_train=X_train,
    ftr_test=X_test,
    tgt_train=y_train,
    tgt_test=y_test
)

## 5. IQR을 이용한 이상치 제거

- 이상치: 전체 데이터의 일반적인 패턴에서 크게 벗어난 값임.
- Q1은 25%, Q2는 중앙값, Q3는 75% 분위수임.
- IQR = Q3 - Q1.
- 하한 = Q1 - 1.5 × IQR, 상한 = Q3 + 1.5 × IQR.
- 범위를 벗어난 데이터를 이상치 후보로 판단함.

✅ 교재 적용 방식

- 상관관계 히트맵으로 Class와 관련성이 큰 피처를 확인함.
- V14, V17이 강한 음의 상관관계를 나타내며 V14를 대상으로 선택함.
- **사기 거래(Class=1)의 V14**에서 분위수를 계산함. 전체 정상·사기 거래에서 계산하는 것이 아님.

❗ 이상치가 반드시 오류는 아님. 사기 검출에서는 극단값이 중요한 사기 패턴일 수 있으므로 제거 이유를 확인해야 함.

In [ ]:
import seaborn as sns

plt.figure(figsize=(9, 9))
corr = card_df.corr()
sns.heatmap(corr, cmap='RdBu')

In [ ]:
import numpy as np

def get_outlier(df=None, column=None, weight=1.5):
    # fraud에 해당하는 column 데이터만 추출, 1/4 분위와 3/4 분위 지점을 np.percentile로 구함.
    fraud = df[df['Class'] == 1][column]
    quantile_25 = np.percentile(fraud.values, 25)
    quantile_75 = np.percentile(fraud.values, 75)

    # IQR을 구하고, IQR에 1.5를 곱해 최댓값과 최솟값 지점 구함.
    iqr = quantile_75 - quantile_25
    iqr_weight = iqr * weight
    lowest_val = quantile_25 - iqr_weight
    highest_val = quantile_75 + iqr_weight

    # 최댓값보다 크거나, 최솟값보다 작은 값을 이상치 데이터로 설정하고 DataFrame index 반환.
    outlier_index = fraud[(fraud < lowest_val) | (fraud > highest_val)].index
    return outlier_index


outlier_index = get_outlier(df=card_df, column='V14', weight=1.5)
print('이상치 데이터 인덱스:', outlier_index)

In [ ]:
# get_preprocessed_df()를 로그 변환 후 V14 피처의 이상치 데이터를 삭제하는 로직으로 변경.
def get_preprocessed_df(df=None):
    df_copy = df.copy()
    amount_n = np.log1p(df_copy['Amount'])
    df_copy.insert(0, 'Amount_Scaled', amount_n)
    df_copy.drop(['Time', 'Amount'], axis=1, inplace=True)

    # 이상치 데이터 삭제하는 로직 추가
    outlier_index = get_outlier(df=df_copy, column='V14', weight=1.5)
    df_copy.drop(outlier_index, axis=0, inplace=True)

    return df_copy


X_train, X_test, y_train, y_test = get_train_test_dataset(card_df)

print('### 로지스틱 회귀 예측 성능 ###')
get_model_train_eval(
    lr_clf,
    ftr_train=X_train,
    ftr_test=X_test,
    tgt_train=y_train,
    tgt_test=y_test
)

print('### LightGBM 예측 성능 ###')
get_model_train_eval(
    lgbm_clf,
    ftr_train=X_train,
    ftr_test=X_test,
    tgt_train=y_train,
    tgt_test=y_test
)

## 6. SMOTE 오버 샘플링

- 소수 클래스의 학습 표본을 합성해 클래스 불균형을 줄임.
- 학습 데이터는 X_train_over / y_train_over로 바꾸고 테스트 데이터는 그대로 사용함.
- 재현율이 좋아져도 정상 거래를 사기로 예측하는 FP가 많아져 정밀도가 낮아질 수 있음.
- 모델별로 효과가 다르므로 정밀도·재현율·F1·AUC를 함께 확인함.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=0)
X_train_over, y_train_over = smote.fit_resample(X_train, y_train)

print('SMOTE 적용 전 학습용 피처/레이블 데이터 세트:', X_train.shape, y_train.shape)
print('SMOTE 적용 후 학습용 피처/레이블 데이터 세트:', X_train_over.shape, y_train_over.shape)
print('SMOTE 적용 후 레이블 값 분포:\n', pd.Series(y_train_over).value_counts())

✅ 정밀도·재현율 곡선

- 예측 확률의 임곗값에 따라 정밀도와 재현율이 어떻게 바뀌는지 확인함.
- 보통 임곗값을 높이면 사기로 판단하는 거래가 감소하여 재현율이 낮아지고 정밀도가 높아지는 경향이 있음. 개별 구간에서 단조롭게 변한다는 보장은 없음.
- 임곗값 조정만으로 항상 좋은 균형을 만들 수 있는 것은 아님.



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.metrics import precision_recall_curve
%matplotlib inline

def precision_recall_curve_plot(y_test, pred_proba_c1):
    # threshold ndarray와 이 threshold에 따른 정밀도, 재현율 ndarray 추출.
    precisions, recalls, thresholds = precision_recall_curve(y_test, pred_proba_c1)

    # X축을 threshold값으로, Y축은 정밀도, 재현율 값으로 각각 Plot 수행. 정밀도는 점선으로 표시
    plt.figure(figsize=(8, 6))
    threshold_boundary = thresholds.shape[0]
    plt.plot(
        thresholds, precisions[0:threshold_boundary],
        linestyle='--', label='precision'
    )
    plt.plot(thresholds, recalls[0:threshold_boundary], label='recall')

    # threshold 값 X축의 Scale을 0.1 단위로 변경
    start, end = plt.xlim()
    plt.xticks(np.round(np.arange(start, end, 0.1), 2))

    # x축, y축 label과 legend, 그리고 grid 설정
    plt.xlabel('Threshold value')
    plt.ylabel('Precision and Recall value')
    plt.legend()
    plt.grid()
    plt.show()

In [ ]:
lr_clf = LogisticRegression(max_iter=1000)

# ftr_train과 tgt_train 인자값이 SMOTE 증식된 X_train_over와 y_train_over로 변경됨에 유의
get_model_train_eval(
    lr_clf,
    ftr_train=X_train_over,
    ftr_test=X_test,
    tgt_train=y_train_over,
    tgt_test=y_test
)

In [ ]:
precision_recall_curve_plot(y_test, lr_clf.predict_proba(X_test)[:, 1])

In [ ]:
lgbm_clf = LGBMClassifier(
    n_estimators=1000,
    num_leaves=64,
    n_jobs=-1,
    boost_from_average=False
)

get_model_train_eval(
    lgbm_clf,
    ftr_train=X_train_over,
    ftr_test=X_test,
    tgt_train=y_train_over,
    tgt_test=y_test
)

## 7. 교재 결과 비교 및 개념 정리

|가공 유형|모델|정밀도|재현율|ROC-AUC|
|---|---|---:|---:|---:|
|원본|로지스틱 회귀|0.8679|0.6216|0.9702|
|원본|LightGBM|0.9573|0.7568|0.9790|
|로그 변환|로지스틱 회귀|0.8812|0.6014|0.9727|
|로그 변환|LightGBM|0.9576|0.7635|0.9796|
|이상치 제거|로지스틱 회귀|0.8750|0.6712|0.9743|
|이상치 제거|LightGBM|0.9603|0.8288|0.9780|
|SMOTE|로지스틱 회귀|0.0542|0.9247|0.9737|
|SMOTE|LightGBM|0.9118|0.8493|0.9814|

✅ 해석

- 데이터 로그 변환만으로 모든 모델의 성능이 좋아지지는 않음.
- 교재에서는 이상치 제거 후 재현율이 개선됨. 다만 테스트 표본이 바뀌므로 공정한 비교를 위해서는 고정 테스트 세트가 필요함.
- SMOTE를 적용한 로지스틱 회귀는 재현율이 높지만 정밀도가 매우 낮아 FP가 많음.
- SMOTE를 적용한 LightGBM도 재현율과 정밀도 사이의 변화가 나타남.
- 불균형 문제의 목표는 단순히 클래스 수를 맞추는 것이 아니라, 오류 비용에 맞는 예측을 하는 것임.
- SMOTE가 항상 재현율을 높이고 정밀도를 낮추는 것은 아님. 데이터와 모델에 따른 결과를 확인해야 함.

